# Silver elhub municipality

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from silver_layer.elhub_municipality import ingest, make_spark
from silver_layer import cache

BRONZE_ROOT = PROJECT_ROOT / "data" / "bronze"
SILVER_ROOT = PROJECT_ROOT / "data" / "silver"
print("Project root:", PROJECT_ROOT)
print("Bronze root exists:", BRONZE_ROOT.exists())

Project root: /home/roland/UiS/rewrite/group14
Bronze root exists: True


## Run the pipeline

Adjust `local[8]` for machine.

In [2]:
spark = make_spark(master="local[8]")
spark.sparkContext.setLogLevel("WARN")
report = ingest(bronze_root=BRONZE_ROOT, silver_root=SILVER_ROOT, spark=spark)
report

26/11/19 10:33:39 WARN Utils: Your hostname, homelab resolves to a loopback address: 127.0.1.1; using 192.168.10.53 instead (on interface lan0)
26/11/19 10:33:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/19 10:33:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Checking Silver cache: /home/roland/UiS/rewrite/group14/data/silver/elhub/municipality_consumption
Extracted 37 CSV files. Starting Spark transformation


Writing 18,842,256 Silver rows to staging directory


26/11/19 10:35:11 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


{
  "dataset": "CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR",
  "status": "success",
  "rows": 18842256,
  "csv_files": 37,
  "silver_path": "/home/roland/UiS/rewrite/group14/data/silver/elhub/municipality_consumption",
  "bronze_archive": "/home/roland/UiS/rewrite/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR_2a7be0d7e979bb07.zip"
}


{'dataset': 'CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR',
 'status': 'success',
 'rows': 18842256,
 'csv_files': 37,
 'silver_path': '/home/roland/UiS/rewrite/group14/data/silver/elhub/municipality_consumption',
 'bronze_archive': '/home/roland/UiS/rewrite/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR_2a7be0d7e979bb07.zip'}

## Inspect Silver output

In [3]:
output = SILVER_ROOT / "elhub" / "municipality_consumption"
print(cache.read_manifest(output))
df = spark.read.parquet(str(output))
df.printSchema()
df.show(10, truncate=False)

{'bronze_archive': '/home/roland/UiS/rewrite/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR_2a7be0d7e979bb07.zip', 'completed_at_utc': '2026-11-19T10:35:18.000713+00:00', 'csv_files': 37, 'dataset': 'CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR', 'fingerprint': {'end_exclusive_local': '2026-01-01', 'source_sha256': '2a7be0d7e979bb07401eb5ce0b93d73d99f7064e106df9d9ab2ee52f12886add', 'start_local': '2024-01-01', 'transformation_version': 'municipality-v2'}, 'rows': 18842256, 'run_id': None}
root
 |-- start_time: timestamp (nullable = true)
 |-- end_time: timestamp (nullable = true)
 |-- municipality_number: string (nullable = true)
 |-- name: string (nullable = true)
 |-- consumption_group: string (nullable = true)
 |-- quantity_kwh: decimal(18,6) (nullable = true)
 |-- metering_point_count: integer (nullable = true)
 |-- last_update_time: timestamp (nullable = true)

+-------------------+-------------------+-------------------+------

In [4]:
from pyspark.sql import functions as F

print("Rows:", df.count())
df.groupBy("consumption_group").count().orderBy("consumption_group").show()
df.select(F.min("start_time").alias("first_utc"), F.max("start_time").alias("last_utc")).show(truncate=False)
df.where(F.col("municipality_number").isin("0000", "0301")).groupBy("municipality_number").count().show()

Rows: 18842256
+-----------------+-------+
|consumption_group|  count|
+-----------------+-------+
|         business|6280752|
|         industry|6280752|
|          private|6280752|
+-----------------+-------+

+-------------------+-------------------+
|first_utc          |last_utc           |
+-------------------+-------------------+
|2023-12-31 23:00:00|2025-12-31 22:00:00|
+-------------------+-------------------+

+-------------------+-----+
|municipality_number|count|
+-------------------+-----+
|               0301|52632|
|               0000|52632|
+-------------------+-----+



# Check that no duplicates exist

In [5]:
from pyspark.sql import functions as F

duplicates = (
    df.groupBy(
        "start_time",
        "municipality_number",
        "consumption_group",
    )
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate keys:", duplicates.count())

26/11/19 10:35:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 10:35:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 10:35:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 10:35:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 10:35:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 10:35:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 10:35:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 10:35:20 WARN RowBasedKeyValueBatch: Calling spill() on RowBasedKeyValueBatch. Will not spill but return 0.
26/11/19 10:35:20 WARN RowBasedKeyValueBatch: Calling spill() on

Duplicate keys: 0


## Confirm cache reuse

In [6]:
second_run = ingest(bronze_root=BRONZE_ROOT, silver_root=SILVER_ROOT, spark=spark)
second_run

Checking Silver cache: /home/roland/UiS/rewrite/group14/data/silver/elhub/municipality_consumption
{
  "dataset": "CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR",
  "status": "cached",
  "rows": 18842256,
  "silver_path": "/home/roland/UiS/rewrite/group14/data/silver/elhub/municipality_consumption",
  "bronze_archive": "/home/roland/UiS/rewrite/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR_2a7be0d7e979bb07.zip"
}


{'dataset': 'CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR',
 'status': 'cached',
 'rows': 18842256,
 'silver_path': '/home/roland/UiS/rewrite/group14/data/silver/elhub/municipality_consumption',
 'bronze_archive': '/home/roland/UiS/rewrite/group14/data/bronze/elhub/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR/CONSUMPTION_PER_GROUP_MUNICIPALITY_HOUR_2a7be0d7e979bb07.zip'}

## Cleanup

In [7]:
spark.stop()